In [0]:
%run "./Util"

In [0]:
#Notebook 02: Reading from Volume and writing to Staging Volume As Parquet and then Bronze Table

from datetime import datetime
from pyspark.sql import functions as F
from datetime import datetime

"""
- Load Raw Data to Bronze Layer
- Reads from Raw_Volume, writes Parquet to Stg_Volume Volume
- Writes to Bronze Table
- Logs results to child metrics metadata table
"""

def file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def read_csv(path):
    return (
        spark.read
        .format("csv")
        .option("header",                  "true")
        .option("inferSchema",             "false")
        .option("multiLine",               "true")
        .option("quote",                   '"')
        .option("escape",                  '"')
        .option("ignoreLeadingWhiteSpace", "true")
        .option("ignoreTrailingWhiteSpace","true")
        .option("encoding",                "UTF-8")
        .load(path)
    )

def write_parquet_staging(df, stg_path, city, batch_id, RUN_TS):
    df_out = (
        df
        .withColumn("_batch_id",      F.lit(batch_id))
        .withColumn("_source_city",   F.lit(city))
        .withColumn("_ingested_at",   F.lit(RUN_TS))
    )
    (df_out
        .write
        .format("parquet")
        .mode("overwrite")            
        .option("compression", "snappy")
        .save(stg_path)
    )

    # Confirm write succeeded by listing the path
    if(file_exists(stg_path)):
        target_df = spark.read.parquet(stg_path)
        target_row_count = target_df.count()
        print(f"{city} - {target_row_count:,} rows written to staging")
    else:
        print(f"ERROR: {city} Staging write failed!!!")
        return 0, None
    
    return target_row_count, target_df



#READING ACTIVE TABLES FROM PARENT METADATA
active_tables_df = spark.sql(f"""
    SELECT DISTINCT table_name
    FROM {CATALOG}.{META_SCHEMA}.{PARENT_META_TABLE}
    WHERE active_flag = 'Y'
""")

active_tables = [row["table_name"] for row in active_tables_df.collect()]
print(f"Active files to process: {active_tables}\n")

#PROCESSING EACH TABLE

for table_name in active_tables:
    try:
        print(f"\nProcessing {table_name} at {datetime.now()}")

        #Step 0: Confirm file exists
        if not file_exists(LANDING_FILE_MAP.get(table_name)):
            print(f"WARN: File not found: {LANDING_FILE_MAP.get(table_name)} at {datetime.now()}")
            #Log Skiiped Log into child metrics
            spark.sql(f"""
                INSERT INTO {CATALOG}.{META_SCHEMA}.{CHILED_META_TABLE} VALUES (
                    '{table_name}',
                    '{datetime.now()}',
                    'SKIPPED',
                    0,
                    0,
                    'N/A',
                    current_date()
                )
            """)
            continue

        #Step 1: Read from Raw Volume
        staging_df = read_csv(LANDING_FILE_MAP.get(table_name))
        source_row_count = staging_df.count()

        #Step 2: Build dynamic timestamp file path        
        RUN_TS      = datetime.now()
        RUN_TS_STR  = RUN_TS.strftime("%Y%m%d_%H%M%S")   # e.g. 20240415_143022
        RUN_DATE    = RUN_TS.strftime("%Y/%m/%d")        # e.g. 2024/04/15
        BATCH_ID    = f"BATCH_{RUN_TS_STR}"              # e.g. BATCH_20240415_143022

        STG_FILE_NAME  = f"{FILE_NAME_MAP.get(table_name).lower() + '_' + RUN_TS_STR}.parquet"
        STG_FILE_PATH  = f"{STAGING_VOLUME_PATH}/{RUN_DATE}/{STG_FILE_NAME}"
        
        #Step 3: Write to STG Volume as Parquet and Verify target row count matches source
        target_row_count, target_df = write_parquet_staging(staging_df, STG_FILE_PATH, table_name, BATCH_ID, RUN_TS)

        #Step 4: 
        status  = "SUCCESS" if source_row_count == target_row_count else "FAILED"

        #Step 5: Log to child metrics metadata
        spark.sql(f"""
            INSERT INTO {CATALOG}.{META_SCHEMA}.{CHILED_META_TABLE} VALUES (
                '{table_name}',
                '{RUN_TS}',
                '{status}',
                {source_row_count},
                {target_row_count},
                '{STG_FILE_PATH}',
                current_date()
            )
        """)
        if status == "SUCCESS":
            print(f"Written to: {STG_FILE_PATH}")
            print(f"{table_name} | {status} | Source: {source_row_count} | Target: {target_row_count} | Batch ID: {BATCH_ID}")
            
            #Rename columns to remove spaces
            target_df = target_df.toDF(*[c.replace(" ", "_") for c in target_df.columns])
            
            df_bronze = (
                target_df
                .withColumn("_bronze_loaded_at", F.lit(datetime.now()))
                .withColumn("_batch_id",         F.lit(BATCH_ID))
            )
            (
            df_bronze.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE_MAP.get(table_name)}")
            )

            print(f"\nWritten to: {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE_MAP.get(table_name)}")
            # print(f"{table_name} | {status} | Source: {source_row_count} | Target: {target_row_count}")
        else:
            print(f"\nERROR: {table_name} | {status} | Source: {source_row_count} | Target: {target_row_count}")

        print("\n--------------------------------------------------------------------------------")
        print("\n--------------------------------------------------------------------------------")
    except Exception as e:
        #Log failure to child metrics
        import traceback
        print(f"{table_name} FAILED: {str(e)}")
        print(traceback.format_exc())
        spark.sql(f"""
            INSERT INTO {CATALOG}.{META_SCHEMA}.{CHILED_META_TABLE} VALUES (
                '{table_name}',
                '{datetime.now()}',
                'FAILED',
                0,
                0,
                'N/A',
                current_date()
            )
    """)

print("\nRaw load completed...")

Active files to process: ['Dallas', 'Chicago']


Processing Dallas at 2026-04-12 22:58:36.492000
Dallas - 78,984 rows written to staging
Written to: /Volumes/Final_Project/Bronze_Zone/vol_stg_files/2026/04/12/dallas_restaurant_and_food_establishment_inspections_20260412_225843.parquet
Dallas | SUCCESS | Source: 78984 | Target: 78984 | Batch ID: BATCH_20260412_225843

Written to: Final_Project.Bronze_Zone.bronze_dallas

--------------------------------------------------------------------------------

--------------------------------------------------------------------------------

Processing Chicago at 2026-04-12 22:59:01.337493
Chicago - 308,357 rows written to staging
Written to: /Volumes/Final_Project/Bronze_Zone/vol_stg_files/2026/04/12/chicago_restaurant_and_food_establishment_inspections_20260412_225906.parquet
Chicago | SUCCESS | Source: 308357 | Target: 308357 | Batch ID: BATCH_20260412_225906

Written to: Final_Project.Bronze_Zone.bronze_chicago

-------------------------------